In [ ]:
# Example 1

import sympy as sp

# Define symbols
x, y = sp.symbols('x y')

# Define function
f = x**3 * y + x**2*y
dfdx = sp.diff(f, x)      # partial derivative of f wrt x
dfdy = sp.diff(f, y)      # partial derivative of f wrt y
gradient = [dfdx, dfdy]


print(dfdx)
print(dfdy)
print(gradient)

In [ ]:
# Example 1

# Evaluate at a point
dfdxv = dfdx.subs({x: 2, y: -2})
dfdyv = dfdy.subs({x: 2, y: -2})
gradientv = [dfdxv, dfdyv]

print(dfdxv)
print(dfdyv)
print([dfdxv, dfdyv])

In [ ]:
# Example 1

# Evaluate at a point
dfdxv1 = dfdx.subs({x: 1.9, y: -2.1})
dfdyv1 = dfdy.subs({x: 1.9, y: -2.1})
print([dfdxv1, dfdyv1])

In [ ]:
# Example 1 by numpy

import numpy as np
from scipy import optimize

def g(x, y):
    return x**3 * y + x**2*y

# Using numpy.gradient for discrete data
x_vals = np.linspace(-10, 10, 51)
y_vals = np.linspace(-10, 10, 51)
print(x_vals)
print(y_vals)

#X, Y = np.meshgrid(x_vals, y_vals)
#Z = g(X, Y)
Z=g(x_vals[:,np.newaxis],y_vals[np.newaxis,:]) # broadcast

# Gradient of the grid
grad_x, grad_y = np.gradient(Z, x_vals, y_vals) 

def g_vector(inputs):
    x, y = inputs
    return g(x, y)

point = [1.9, -2.1]
epsilon = 1e-6  # Step size for finite differences
gradient = optimize.approx_fprime(point, g_vector, epsilon)
print(f"Numerical gradient at (1.9, -2.1): {gradient}")

In [ ]:
#meshgrid 和 broadcast 性能测试
import numpy as np
from scipy import optimize
import time
import sys

def g(x, y):
    return x**3 * y + x**2*y

# --- 你原本的代码逻辑 ---
x_vals = np.linspace(-10, 10, 51)
y_vals = np.linspace(-10, 10, 51)

# 使用 broadcast 计算 Z
Z = g(x_vals[:, np.newaxis], y_vals[np.newaxis, :]) 

# 计算梯度
grad_x, grad_y = np.gradient(Z, x_vals, y_vals) 

def g_vector(inputs):
    x, y = inputs
    return g(x, y)

point = [1.9, -2.1]
epsilon = 1e-6  
gradient = optimize.approx_fprime(point, g_vector, epsilon)
print(f"Numerical gradient at (1.9, -2.1): {gradient}\n")


# ==========================================
# --- 性能极限测试方案：Meshgrid vs Broadcast ---
# ==========================================
print("-" * 40)
print("开始极限性能测试 (5000 x 5000 网格)...")
N = 5000
x_test = np.linspace(-10, 10, N)
y_test = np.linspace(-10, 10, N)

# 1. 测试 np.meshgrid 显式网格
start_time = time.perf_counter()
# 注意：这里必须加 indexing='ij'，否则算出来的矩阵形状和 broadcast 是转置关系
X_mesh, Y_mesh = np.meshgrid(x_test, y_test, indexing='ij') 
Z_mesh = g(X_mesh, Y_mesh)
mesh_time = time.perf_counter() - start_time

# 计算 meshgrid 产生的冗余内存 (X_mesh, Y_mesh 加上结果 Z_mesh)
mesh_memory_mb = (sys.getsizeof(X_mesh) + sys.getsizeof(Y_mesh) + sys.getsizeof(Z_mesh)) / (1024 * 1024)


# 2. 测试 np.newaxis 隐式广播
start_time = time.perf_counter()
# x_test[:, np.newaxis] 只是改变视图，几乎不占额外内存
Z_broad = g(x_test[:, np.newaxis], y_test[np.newaxis, :])
broad_time = time.perf_counter() - start_time

# 计算 broadcast 的内存 (仅计算两个一维视图和最终结果 Z_broad)
broad_memory_mb = (sys.getsizeof(x_test[:, np.newaxis]) + sys.getsizeof(y_test[np.newaxis, :]) + sys.getsizeof(Z_broad)) / (1024 * 1024)


# --- 打印对比结果 ---
print(f"1. Meshgrid 耗时: {mesh_time:.4f} 秒, 占用内存: {mesh_memory_mb:.2f} MB")
print(f"2. Broadcast 耗时: {broad_time:.4f} 秒, 占用内存: {broad_memory_mb:.2f} MB")
print(f"数据结果是否完全一致？: {np.allclose(Z_mesh, Z_broad)}")
print("-" * 40)

In [ ]:
from scipy import integrate as inte

def f(x):
    return x**2 + np.sin(x)

# Single integration
result, error = inte.quad(f, 0, 1)
print(f"Result: {result}, Error estimate: {error}")

In [ ]:
# Example 4 Define CDF

import numpy as np

def f(x):
    """
    Define piecewise function by np.piecewise
    """
    x = np.asarray(x)   # change the data type of x from floating scalar to numpy array.
    
    # Define conditions and the corresponding formulae
    conditions = [
        x < 0,
        (x >= 0) & (x <= 1),
        x > 1
    ]
    
    functions = [
        lambda x: 0,
        lambda x: (2 * x) / (x + 1),
        lambda x: 1
    ]
    
    result = np.piecewise(x, conditions, functions)
    # The data typs of result is numpy.ndarray, n dimensional array.
    
    # Return the result as scalar, instead of array
    if np.isscalar(x):
       return result.item()
    return result

# Test
x_vals = np.array([-1, -0.5, 0, 0.25, 0.5, 0.75, 1, 1.5, 2])
print("by np.piecewise:")
for x in x_vals:
    print(f"f({x:.2f}) = {f(x):.3f}")

In [ ]:
# Example 4 (a)

print(f"P(X<2) = {f(2)}")


In [ ]:
# Example 4 (b)

print(f"P(0.5<X<0.7) = {f(0.7)-f(0.5)}")

In [ ]:
# Example 4 (c)

import sympy as sp

x = sp.symbols('x')

# Define CDF
f = sp.Piecewise(
    (0, x < 0),
    (1, x > 1),
    ((2 * x) / (x + 1), True))


f_prime = sp.diff(f, x)

print("CDF f(x) =")
print(f)
print("\nPDF is")
print(f_prime)

In [ ]:
# Example 5 (a) and (b)

import numpy as np
from scipy import integrate


def f(x):
    return (6 * x * (5 - x)) / 125

def f1(x):
    return (6 * x**2 * (5 - x)) / 125

def f2(x):
    return (6 * x**3 * (5 - x)) / 125


result, error = integrate.quad(f, 3, 5)
print(f"P(X > 3) = {result}")
print(f"Calculationn Error = {error}")

exp, error1 = integrate.quad(f1, 0, 5)
print(f"\nThe expected value is {exp}")

Exp, error2 = integrate.quad(f2, 0, 5)
Var = Exp - exp**2 
print(f"\nThe variance is {Var}")


In [ ]:
# Example 5 (c)

import sympy as sp

# Follow Example 4
x = sp.symbols('x')


f = sp.Piecewise(
    (0, x < 0),
    (0, x > 5),
    ((6 * x * (5 - x)) / 125, True)  # 0 ≤ x ≤ 5
)

print(f"PDF = {f}")



F = sp.integrate(f, x)
print(f"\nCDF as ∫f(x)dx = {F}")


In [ ]:
import numpy as np

def f(x):
    x = np.asarray(x)   # change the data type of x from
                        # floating scalar to numpy array.
    # Define conditions and the corresponding formulae
    conditions = [  x < 0,  x >= 0 ]
    functions = [  lambda x: 0, lambda x: 2 * x    ]
    result = np.piecewise(x, conditions, functions)
    # Return the result as scalar, instead of array
    if np.isscalar(x):
       return result.item()
    return result

from scipy import integrate


result, error = integrate.quad(f, 0, 2)
print(f"int f from 0 to 2 = {result}")
    

In [ ]:
import numpy as np

S = np.random.normal(1, 2, 1000)

print(len(S))
print(np.mean(S))
print(np.std(S))
print("Ten elements in sample S:")
print(S[:10])

In [ ]:
import mpmath as mp
import math

mp.dps = 50

e_mp = mp.exp(1)
print(mp.nstr(e_mp, 50))   # 输出：2.71828182845904523536

print(f"{math.exp(1):.50f}")